In [16]:
import pandas as pd
import numpy as np
from math import log2
from math import sqrt
from numpy import asarray
import re
import warnings
warnings.filterwarnings('ignore')


In [17]:
def replace_items(text):
    #text = text.astype(str)
    text = text.lower().strip()

    punc = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''

    for ele in text:
        if ele in punc:
            text = text.replace(ele, "")
    #text = text.replace('not sexist', 0)
    #text = text.replace('sexist', 1)
    return text

In [18]:
data = pd.read_csv('../data/entropy_data/flan2.csv', index_col=[0])
data

,uniqueID,trigger_event,label,roleplay,content,zero-shot,few-shot
0,poli_ukdata_24-0,political,1,"('not sexist', 0.9166759)","('sexist', 0.9298217)","('sexist', 0.9226115)","('sexist', 0.94947547)"
1,poli_ukdata_24-1,political,0,"('not sexist', 0.9407921)","('sexist', 0.9165034)","('sexist', 0.971445)","('sexist', 0.9745518)"
2,poli_ukdata_24-2,political,0,"('not sexist', 0.988616)","('not sexist', 0.99873954)","('not sexist', 0.999465)","('not sexist', 0.9994353)"
3,poli_ukdata_24-3,political,0,"('not sexist', 0.9613606)","('not sexist', 0.9449113)","('not sexist', 0.969962)","('not sexist', 0.984799)"
4,poli_ukdata_24-4,political,0,"('not sexist', 0.97983086)","('not sexist', 0.98537827)","('not sexist', 0.99379426)","('not sexist', 0.99710214)"
...,...,...,...,...,...,...,...
1268,poli_ukdata_24-1268,political,0,"('not sexist', 0.99553406)","('not sexist', 0.9910835)","('not sexist', 0.9970583)","('not sexist', 0.99882257)"
1269,poli_ukdata_24-1269,political,0,"('not sexist', 0.9823745)","('not sexist', 0.993674)","('not sexist', 0.9942519)","('not sexist', 0.9948062)"
1270,poli_ukdata_24-1270,political,0,"('not sexist', 0.98172885)","('not sexist', 0.9963051)","('not sexist', 0.99849415)","('not sexist', 0.9986372)"
1271,poli_ukdata_24-1271,political,0,"('not sexist', 0.9883831)","('not sexist', 0.9962468)","('not sexist', 0.998552)","('not sexist', 0.9989954)"


In [19]:
data['roleplay'] = data['roleplay'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data['roleplay_pred'] = data['roleplay'].str[0]
data['roleplay_prob'] = data['roleplay'].str[1]
data = data.drop(columns='roleplay')
data['roleplay_prob'] = data['roleplay_prob'].astype('str').apply(lambda x: float(x.strip()))
data['roleplay_pred'] = data['roleplay_pred'].apply(lambda x: replace_items(x)).replace({'not sexist': 0, 'sexist': 1})


data['content'] = data['content'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data['content_pred'] = data['content'].str[0]
data['content_prob'] = data['content'].str[1]
data = data.drop(columns='content')
data['content_prob'] = data['content_prob'].astype('str').apply(lambda x: float(x.strip()))
data['content_pred'] = data['content_pred'].apply(lambda x: replace_items(x)).replace({'not sexist': 0, 'sexist': 1})

data['zero-shot'] = data['zero-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data['zero-shot_pred'] = data['zero-shot'].str[0]
data['zero-shot_prob'] = data['zero-shot'].str[1]
data = data.drop(columns='zero-shot')
data['zero-shot_prob'] = data['zero-shot_prob'].astype('str').apply(lambda x: float(x.strip()))
data['zero-shot_pred'] = data['zero-shot_pred'].apply(lambda x: replace_items(x)).replace({'not sexist': 0, 'sexist': 1})

data['few-shot'] = data['few-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data['few-shot_pred'] = data['few-shot'].str[0]
data['few-shot_prob'] = data['few-shot'].str[1]
data = data.drop(columns='few-shot')
data['few-shot_prob'] = data['few-shot_prob'].astype('str').apply(lambda x: float(x.strip()))
data['few-shot_pred'] = data['few-shot_pred'].apply(lambda x: replace_items(x)).replace({'not sexist': 0, 'sexist': 1})

In [20]:
data = data.dropna(subset=['roleplay_prob', 'content_prob', 'zero-shot_prob', 'few-shot_prob'], how = 'all').reset_index(drop=True)
data

,uniqueID,trigger_event,label,roleplay_pred,roleplay_prob,content_pred,content_prob,zero-shot_pred,zero-shot_prob,few-shot_pred,few-shot_prob
0,poli_ukdata_24-0,political,1,0,0.916676,1,0.929822,1,0.922612,1,0.949475
1,poli_ukdata_24-1,political,0,0,0.940792,1,0.916503,1,0.971445,1,0.974552
2,poli_ukdata_24-2,political,0,0,0.988616,0,0.998740,0,0.999465,0,0.999435
3,poli_ukdata_24-3,political,0,0,0.961361,0,0.944911,0,0.969962,0,0.984799
4,poli_ukdata_24-4,political,0,0,0.979831,0,0.985378,0,0.993794,0,0.997102
...,...,...,...,...,...,...,...,...,...,...,...
1267,poli_ukdata_24-1268,political,0,0,0.995534,0,0.991084,0,0.997058,0,0.998823
1268,poli_ukdata_24-1269,political,0,0,0.982375,0,0.993674,0,0.994252,0,0.994806
1269,poli_ukdata_24-1270,political,0,0,0.981729,0,0.996305,0,0.998494,0,0.998637
1270,poli_ukdata_24-1271,political,0,0,0.988383,0,0.996247,0,0.998552,0,0.998995


In [21]:
data['label_match_roleplay'] = data.apply(lambda row: row['label'] == row['roleplay_pred'], axis=1)
data['label_match_content'] = data.apply(lambda row: row['label'] == row['content_pred'], axis=1)
data['label_match_zero'] = data.apply(lambda row: row['label'] == row['zero-shot_pred'], axis=1)
data['label_match_few'] = data.apply(lambda row: row['label'] == row['few-shot_pred'], axis=1)
data

,uniqueID,trigger_event,label,roleplay_pred,roleplay_prob,content_pred,content_prob,zero-shot_pred,zero-shot_prob,few-shot_pred,few-shot_prob,label_match_roleplay,label_match_content,label_match_zero,label_match_few
0,poli_ukdata_24-0,political,1,0,0.916676,1,0.929822,1,0.922612,1,0.949475,False,True,True,True
1,poli_ukdata_24-1,political,0,0,0.940792,1,0.916503,1,0.971445,1,0.974552,True,False,False,False
2,poli_ukdata_24-2,political,0,0,0.988616,0,0.998740,0,0.999465,0,0.999435,True,True,True,True
3,poli_ukdata_24-3,political,0,0,0.961361,0,0.944911,0,0.969962,0,0.984799,True,True,True,True
4,poli_ukdata_24-4,political,0,0,0.979831,0,0.985378,0,0.993794,0,0.997102,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267,poli_ukdata_24-1268,political,0,0,0.995534,0,0.991084,0,0.997058,0,0.998823,True,True,True,True
1268,poli_ukdata_24-1269,political,0,0,0.982375,0,0.993674,0,0.994252,0,0.994806,True,True,True,True
1269,poli_ukdata_24-1270,political,0,0,0.981729,0,0.996305,0,0.998494,0,0.998637,True,True,True,True
1270,poli_ukdata_24-1271,political,0,0,0.988383,0,0.996247,0,0.998552,0,0.998995,True,True,True,True


In [22]:
data = data.drop(columns = ['label', 'trigger_event', 'roleplay_pred', 'content_pred', 'zero-shot_pred', 'few-shot_pred'])
data

,uniqueID,roleplay_prob,content_prob,zero-shot_prob,few-shot_prob,label_match_roleplay,label_match_content,label_match_zero,label_match_few
0,poli_ukdata_24-0,0.916676,0.929822,0.922612,0.949475,False,True,True,True
1,poli_ukdata_24-1,0.940792,0.916503,0.971445,0.974552,True,False,False,False
2,poli_ukdata_24-2,0.988616,0.998740,0.999465,0.999435,True,True,True,True
3,poli_ukdata_24-3,0.961361,0.944911,0.969962,0.984799,True,True,True,True
4,poli_ukdata_24-4,0.979831,0.985378,0.993794,0.997102,True,True,True,True
...,...,...,...,...,...,...,...,...,...
1267,poli_ukdata_24-1268,0.995534,0.991084,0.997058,0.998823,True,True,True,True
1268,poli_ukdata_24-1269,0.982375,0.993674,0.994252,0.994806,True,True,True,True
1269,poli_ukdata_24-1270,0.981729,0.996305,0.998494,0.998637,True,True,True,True
1270,poli_ukdata_24-1271,0.988383,0.996247,0.998552,0.998995,True,True,True,True


In [23]:
data.to_csv('../data/entropy_data/hypothesis_data.csv')